# Логистическая регрессия с кросс-валидацией

Модель использует L2-регуляризацию и обучается на проверенных и дополнительных метках с весами. Оцениваем прогнозы на пяти внешних фолдах. Порог каждого фолда подбираем по F1 на его обучающей части. Для заблокированных объявлений всегда ставим `is_deal = 0`.

## 1. Загрузить данные

Используем `features.csv` из `feature_engineering.ipynb`. Метки и признаки сопоставляем по `listing_id`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [ ]:
FEATURES_PATH = Path("data_by_dewallen/features.csv")
VERIFIED_PATH = Path("data/verified.csv")
EXTENDED_VERIFIED_PATH = Path("extend_verified.csv")
LISTINGS_PATH = Path("data/listings.csv")
OUTPUT_PATH = Path("submission.csv")

CV_SEEDS = [11, 23, 42, 71, 101, 13, 29, 59, 83, 107]
THRESHOLDS = np.round(np.arange(0.20, 0.71, 0.01), 2)
INNER_SEED = 42

In [ ]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")
listings = pd.read_csv(LISTINGS_PATH, usecols=["listing_id", "close_type"]).drop_duplicates()
listings = listings.set_index("listing_id")

assert listings.index.is_unique and features.index.is_unique
assert set(features.index) == set(listings.index), "Пересчитайте признаки для всех объявлений"
assert verified.index.is_unique, "Повторяются ID в проверенной разметке"
assert verified.index.isin(features.index).all()
assert verified["is_deal"].isin([0, 1]).all(), "Некорректные метки в verified.csv"
assert features.columns.is_unique

feature_columns = features.columns.difference(["seller_id", "is_deal", "weight"], sort=False)
X = features[feature_columns]
y = verified["is_deal"]

print(f"Объявлений в матрице: {len(X)}")
print(f"Признаков: {X.shape[1]}")
print(f"Объявлений с проверенными метками: {len(y)}")

### Дополнительные метки

Для объявлений из `verified.csv` берём проверенный ответ с весом 1. Из `extend_verified.csv` берём дополнительные метки и исключаем строки с исходным весом 0.8. Для остальных дополнительных меток используем вес 0.5. ID из проверенной разметки исключаем из дополнительных меток, чтобы проверочные объявления не попадали в обучение через другой файл.

In [ ]:
extended = pd.read_csv(
    EXTENDED_VERIFIED_PATH,
    usecols=["listing_id", "is_deal", "weight"],
    index_col="listing_id",
)
assert extended.index.is_unique, "Повторяются ID в расширенной разметке"
assert extended.index.notna().all(), "В расширенной разметке есть пустые ID"

extra_labels = extended.loc[
    extended["weight"].ne(0.8) & ~extended.index.isin(verified.index)
].assign(weight=0.5)
assert extra_labels.index.isin(X.index).all(), "Для дополнительных меток нет признаков"
assert extra_labels["is_deal"].isin([0, 1]).all(), "Некорректные дополнительные метки"
assert extra_labels["weight"].gt(0).all() and np.isfinite(extra_labels["weight"]).all()

print("Дополнительных меток:", len(extra_labels))
display(extra_labels.groupby(["is_deal", "weight"]).size())

## 2. Создать модель

В числовом блоке заменяем бесконечности пропусками, заполняем пропуски медианой и стандартизируем значения. В категориальном блоке заполняем пропуски самым частым значением и применяем one-hot. Каждый фолд получает свои параметры подготовки, рассчитанные на его обучающей части.

Логистическая регрессия использует L2-регуляризацию с `C=1`.

In [ ]:
def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, make_column_selector(dtype_include=["number", "bool"])),
    ("categorical", categorical_pipeline, make_column_selector(dtype_exclude=["number", "bool"])),
])
model = make_pipeline(
    preprocessor,
    LogisticRegression(C=1.0, max_iter=2000),
)

## 3. Оценить качество на пяти фолдах

Внутренние и внешние фолды строим по проверенным меткам. В каждом обучении к проверенным строкам текущего фолда добавляем дополнительные метки с их весами. По внутренним отложенным прогнозам выбираем порог с наибольшим F1. При равенстве берём порог ближе к 0,5.

In [ ]:
blocked_ids = listings.index[listings["close_type"] == "заблокировано"]

In [ ]:
def fit_model(model, X, y_train, extra_labels):
    train_labels = pd.concat([
        y_train.to_frame().assign(weight=1.0),
        extra_labels,
    ])
    X_fit = X.loc[train_labels.index]
    return clone(model).fit(
        X_fit,
        train_labels["is_deal"],
        logisticregression__sample_weight=train_labels["weight"],
    )

In [ ]:
def choose_threshold(model, X, y_train, extra_labels, blocked_ids):
    inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=INNER_SEED)
    X_train = X.loc[y_train.index]
    train_probabilities = np.zeros(len(y_train))

    for train, test in inner_cv.split(X_train, y_train):
        inner_model = fit_model(model, X, y_train.iloc[train], extra_labels)
        train_probabilities[test] = inner_model.predict_proba(X_train.iloc[test])[:, 1]

    train_probabilities[X_train.index.isin(blocked_ids)] = 0

    return max(
        THRESHOLDS,
        key=lambda threshold: (
            f1_score(y_train, train_probabilities > threshold),
            -abs(threshold - 0.5),
        ),
    )

In [ ]:
def evaluate_model(model, X, y, extra_labels, blocked_ids, seed):
    """Прогнозируем каждое объявление на фолде, где модель его не видела."""
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    X_verified = X.loc[y.index]
    fold_metrics = []
    fold_predictions = []
    for fold, (train, test) in enumerate(cv.split(X_verified, y), start=1):
        y_fit = y.iloc[train]
        threshold = choose_threshold(model, X, y_fit, extra_labels, blocked_ids)
        fold_model = fit_model(model, X, y_fit, extra_labels)

        y_test = y.iloc[test]
        p_deal = pd.Series(
            fold_model.predict_proba(X_verified.iloc[test])[:, 1],
            index=y_test.index,
        )
        p_deal.loc[y_test.index.intersection(blocked_ids)] = 0
        labels = (p_deal > threshold).astype(int)

        fold_metrics.append({
            "Фолд": fold,
            "Порог": threshold,
            "f1": f1_score(y_test, labels),
            "precision": precision_score(y_test, labels),
            "recall": recall_score(y_test, labels),
            "roc_auc": roc_auc_score(y_test, p_deal),
            "accuracy": accuracy_score(y_test, labels),
        })
        fold_predictions.append(pd.DataFrame({
            "fold": fold,
            "is_deal_true": y_test,
            "is_deal_pred": labels,
            "p_deal": p_deal,
        }))

    oof_predictions = pd.concat(fold_predictions).loc[y.index]
    assert oof_predictions.index.equals(y.index), "Нужен один прогноз на каждое объявление"
    by_fold = pd.DataFrame(fold_metrics).set_index("Фолд")
    return oof_predictions, by_fold

def category_deal_shares(predictions, category):
    return predictions.join(category).groupby("category")[
        ["is_deal_true", "is_deal_pred"]
    ].mean()

In [ ]:
oof_predictions, by_fold = evaluate_model(
    model, X, y, extra_labels, blocked_ids, seed=42,
)
metrics = by_fold.drop(columns="Порог")
summary = pd.DataFrame({
    "Среднее": metrics.mean(),
    "Стандартное отклонение": metrics.std(ddof=0),
})
print(f"Проверено объявлений: {len(oof_predictions)} из {len(verified)}")
print(f"Заблокированных в проверке: {oof_predictions.index.isin(blocked_ids).sum()}")
display(by_fold.round(4))
display(summary.round(4))

# Доли сделок в категориях по отложенным прогнозам.
category_shares = category_deal_shares(oof_predictions, features["category"])
cat_share_mae = (category_shares["is_deal_true"] - category_shares["is_deal_pred"]).abs().mean()

print("F1 по всем проверочным прогнозам:", round(
    f1_score(oof_predictions["is_deal_true"], oof_predictions["is_deal_pred"]), 4
))
print("CatShareMAE, п. п.:", round(100 * cat_share_mae, 2))
display(category_shares.round(4))

### Проверка новых признаков

На одинаковых разбиениях сравниваем группы признаков и вклад сигнала из разных чатов. Для итоговой модели используем все признаки.

In [ ]:
dialogue_columns = [
    "has_arrival_without_later_refusal",
    "has_mutual_meeting",
    "has_handoff_thanks",
    "buyer_refusal_share",
    "has_seller_sold",
]
buyer_contact_columns = [
    "buyers_with_chat_and_phone",
    "repeat_contact_buyer_share",
    "phone_viewers_last_24h",
]
cross_chat_column = "has_unavailable_after_arrival_in_other_chat"
base_features = X.drop(columns=cross_chat_column)

feature_sets = {
    "Базовые признаки": base_features.drop(columns=dialogue_columns + buyer_contact_columns),
    "С этапами разговора": base_features.drop(columns=buyer_contact_columns),
    "С контактами покупателей": base_features.drop(columns=dialogue_columns),
    "Без сигнала из разных чатов": base_features,
    "С сигналом из разных чатов": X,
}

In [ ]:
comparison_metrics = []
for seed in CV_SEEDS:
    for name, training_features in feature_sets.items():
        validation_predictions, _ = evaluate_model(
            model, training_features, y, extra_labels, blocked_ids, seed,
        )
        shares = category_deal_shares(validation_predictions, features["category"])
        share_error = (shares["is_deal_true"] - shares["is_deal_pred"]).abs().mean()

        comparison_metrics.append({
            "Набор": name,
            "Разбиение": seed,
            "F1": f1_score(
                validation_predictions["is_deal_true"],
                validation_predictions["is_deal_pred"],
            ),
            "CatShareMAE, п. п.": 100 * share_error,
        })

comparison_results = pd.DataFrame(comparison_metrics)
display(comparison_results.round(4))
display(comparison_results.groupby("Набор", sort=False)[
    ["F1", "CatShareMAE, п. п."]
].mean().round(4))

### Сигнал из разных чатов на проверенных метках

Повторяем сравнение на одних проверенных метках, чтобы сопоставить результат с исходным экспериментом. Используем те же разбиения и подбираем порог внутри обучения каждого внешнего фолда.

In [ ]:
cross_chat_comparison = []
for seed in CV_SEEDS:
    for training_features in [base_features, X]:
        validation_predictions, _ = evaluate_model(
            model, training_features, y, extra_labels.iloc[:0], blocked_ids, seed,
        )
        shares = category_deal_shares(validation_predictions, features["category"])
        share_error = (shares["is_deal_true"] - shares["is_deal_pred"]).abs().mean()
        cross_chat_comparison.append({
            "Разбиение": seed,
            "Сигнал из разных чатов": cross_chat_column in training_features,
            "F1": f1_score(
                validation_predictions["is_deal_true"],
                validation_predictions["is_deal_pred"],
            ),
            "CatShareMAE, п. п.": 100 * share_error,
        })

verified_comparison = pd.DataFrame(cross_chat_comparison)
display(verified_comparison.groupby("Сигнал из разных чатов")[["F1", "CatShareMAE, п. п."]].mean().round(4))
f1_by_seed = verified_comparison.pivot(index="Разбиение", columns="Сигнал из разных чатов", values="F1")
print("Разбиений с ростом F1:", (f1_by_seed[True] > f1_by_seed[False]).sum())

## 4. Обучить итоговую модель и получить прогнозы

Подбираем порог по четырём фолдам проверенной разметки с дополнительными метками в обучении. Итоговую модель обучаем на всех проверенных и дополнительных метках и применяем этот порог к прогнозам по всем объявлениям.

In [ ]:
threshold = choose_threshold(model, X, y, extra_labels, blocked_ids)
print("Итоговый порог:", round(threshold, 2))

final_model = fit_model(model, X, y, extra_labels)
probabilities = final_model.predict_proba(X)[:, 1]
predictions = X.index.to_frame(index=False).assign(
    is_deal=(probabilities > threshold).astype(int),
)

## 5. Сохранить CSV

Оставляем прогнозы модели для незаблокированных объявлений. Для заблокированных ставим `is_deal = 0`.

In [ ]:
submission = predictions.copy()
submission.loc[submission["listing_id"].isin(blocked_ids), "is_deal"] = 0

assert submission.columns.tolist() == ["listing_id", "is_deal"]
assert submission["listing_id"].is_unique
assert set(submission["listing_id"]) == set(listings.index)
assert submission["is_deal"].isin([0, 1]).all()

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Сохранено {len(submission)} объявлений в {OUTPUT_PATH}")
submission.head()